# Predict, Ship, Compete
## From SQL to Live A/B Test

**Your mission**: Build a model that decides which ads to show to which users to **maximize revenue** (not just clicks).

Your model will be deployed in a live A/B test against other teams. The team that generates the most revenue per impression wins.

---

> **Setup requirement: Python 3.12.** Models pickled under a different Python minor version (3.11, 3.13, ...) cannot be loaded by the server and will fail at upload with cryptic errors. If you're not on 3.12, create a fresh environment first: `conda create -n mayday python=3.12` (or `python3.12 -m venv .venv`). Then `pip install -r requirements.txt`.

**Server URL is set in the next cell** — pick a unique team name.

In [ ]:
import sys
if sys.version_info[:2] != (3, 12):
    print(f'WARNING: this workshop expects Python 3.12 — you are on '
          f'{sys.version_info.major}.{sys.version_info.minor}. Model upload '
          f'may fail with pickle errors. See the markdown cell above for setup.')

SERVER = 'https://lse-mayday.onrender.com'
TEAM_NAME = 'your-team-name'  # Pick a unique team name

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cloudpickle
import time
import io
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
def query(sql, limit=50000):
    """Run a SQL query against the workshop database."""
    resp = requests.post(f"{SERVER}/api/sql", json={"query": sql, "limit": limit})
    resp.raise_for_status()
    data = resp.json()
    return pd.DataFrame(data["rows"], columns=data["columns"])

### Register your team

In [ ]:
resp = requests.post(f"{SERVER}/api/teams/{TEAM_NAME}/register",
                     json={"members": ["Alice", "Bob"]})  # Put your names here
print(resp.json())

---

# Phase 1: Explore the Data (45 min)

You have access to an e-commerce ad database with four tables:
- **users** — 10,000 users with demographics and behavior
- **ads** — 200 ad creatives with product and creative metadata
- **impressions** — 500,000 ad impressions (did the user click?)
- **conversions** — purchases that followed a click

You can also explore interactively at the **SQL Explorer**: `{SERVER}/sql`

## 1.1 Get the lay of the land

In [ ]:
# Check what we're working with
schema = requests.get(f"{SERVER}/api/schema").json()
for table, info in schema["tables"].items():
    cols = [c["name"] for c in info["columns"]]
    print(f"\n{table} ({info['row_count']:,} rows):")
    print(f"  Columns: {', '.join(cols)}")

In [ ]:
# Overall funnel metrics
query("""
SELECT
  COUNT(*) as impressions,
  SUM(clicked) as clicks,
  ROUND(AVG(clicked) * 100, 2) as ctr_pct,
  (SELECT COUNT(*) FROM conversions) as conversions,
  (SELECT ROUND(SUM(revenue), 2) FROM conversions) as total_revenue
FROM impressions
""")

## 1.2 Investigate: Are all clicks equally valuable?

**Key question**: If you optimize purely for CTR (click-through rate), will you also maximize revenue?

In [ ]:
# CTR and revenue by user segments
# Try different groupings: loyalty_tier, device_type, age_group, etc.

segment_stats = query("""
SELECT
  u.loyalty_tier,
  u.device_type,
  COUNT(*) as impressions,
  SUM(i.clicked) as clicks,
  ROUND(AVG(i.clicked) * 100, 2) as ctr_pct,
  COUNT(c.conversion_id) as conversions,
  ROUND(SUM(c.revenue), 2) as total_revenue,
  ROUND(SUM(c.revenue) / COUNT(*), 4) as revenue_per_impression
FROM impressions i
JOIN users u ON i.user_id = u.user_id
LEFT JOIN conversions c ON i.impression_id = c.impression_id
GROUP BY u.loyalty_tier, u.device_type
ORDER BY revenue_per_impression DESC
""")
segment_stats

In [ ]:
# YOUR EXPLORATION: What patterns do you see?
# - Which segments have high CTR but low revenue?
# - Which segments have low CTR but high revenue per impression?
# - What does this mean for your modeling strategy?
#
# Write your own queries below:



In [ ]:
# Hint: do clickbait-y ads generate more revenue?
# Try grouping by clickbait score buckets and comparing CTR with
# revenue per impression. The pattern might surprise you.

### Insight check

Before moving on, make sure you can answer:
1. Do high-CTR user segments also have the highest revenue per impression?
2. Do high-clickbait ads generate more revenue than high-quality ads?
3. What is the conversion rate (given click) for different user types?

**The answer to these questions should inform what your model optimizes for.**

---

# Phase 2: Build Your Model (75 min)

## 2.1 Pull training data

Pull the full dataset with user, ad, and context features joined together.

In [ ]:
# Pull a sample of the joined dataset in chunks.
# 100k rows is plenty to learn the patterns and trains a strong model.
# The full dataset is 500k — increase max_rows if you want more, but bear
# in mind every team is hitting the same server.

def pull_training_data(max_rows=100_000):
    batch_size = 50000
    all_data = []
    offset = 0
    while offset < max_rows:
        batch = min(batch_size, max_rows - offset)
        df = query(f"""
        SELECT
            i.impression_id,
            -- User features
            u.age_group, u.gender, u.device_type, u.region,
            u.account_age_days, u.past_purchases, u.avg_order_value,
            u.sessions_per_week, u.loyalty_tier,
            -- Ad features
            a.category, a.ad_format, a.product_price, a.discount_pct,
            a.creative_quality_score, a.headline_clickbait_score,
            a.brand_familiarity,
            -- Context features
            i.page_type, i.position, i.hour_of_day, i.day_of_week,
            i.session_depth,
            -- Targets
            i.clicked,
            CASE WHEN c.conversion_id IS NOT NULL THEN 1 ELSE 0 END as converted,
            COALESCE(c.revenue, 0) as revenue
        FROM impressions i
        JOIN users u ON i.user_id = u.user_id
        JOIN ads a ON i.ad_id = a.ad_id
        LEFT JOIN conversions c ON i.impression_id = c.impression_id
        ORDER BY i.impression_id
        LIMIT {batch} OFFSET {offset}
        """, limit=batch)

        if len(df) == 0:
            break
        all_data.append(df)
        offset += batch
        print(f"  Pulled {offset:,} rows...")

    return pd.concat(all_data, ignore_index=True)

print("Pulling training data...")
df = pull_training_data()
print(f"\nTotal: {len(df):,} rows")
df.head()

In [ ]:
# Worth checking before you train anything:
#   - What's the overall CTR? CVR given click?
#   - What's average revenue per impression? Per conversion?
#   - Does df.head() / df.describe() look sensible?
#   - Any nulls or weird values?

## 2.2 Feature engineering

Prepare features for your model. The model will need to accept a DataFrame with these columns during inference.

In [ ]:
def prepare_features(df):
    """Prepare features for modelling.

    The same function will be used at inference time — it must work on
    a single row or a batch and produce the same column order both times.
    """
    # e.g. make categorical variables (age_group, region, category, ...)
    # into dummies; pass numeric ones through as-is. Return a DataFrame
    # ready for your model.
    raise NotImplementedError

X = prepare_features(df)
print(f"Feature matrix: {X.shape}")

## 2.3 Choose your target

This is the most important decision. What should your model predict?

The A/B test scores you on **revenue per impression** — but `revenue`
itself is sparse and noisy, so regressing on it directly isn't always
the best move. Think carefully about what target most closely tracks
the metric you'll actually be judged on.

In [ ]:
# Pick a target (or build one):
y_click = df['clicked'].values
y_revenue = df['revenue'].values

# Hint: revenue per impression is a chain of conditional events.
# What has to happen first for revenue to exist at all? Then what?
# How would you model each step?

In [ ]:
# Split data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_click, test_size=0.2, random_state=42  # Change target as needed
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2.4 Train your model

Start simple and iterate. Remember: your model needs to be fast at inference time (Phase 3).

In [ ]:
# Example: Logistic Regression (fast, interpretable baseline)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict_proba(X_test)[:, 1]
print(f"Logistic Regression:")
print(f"  AUC: {roc_auc_score(y_test, y_pred_lr):.4f}")
print(f"  Log Loss: {log_loss(y_test, y_pred_lr):.4f}")

In [ ]:
# Other classifiers / regressors (RandomForest, LightGBM, XGBoost,
# neural nets) are all fine — the server accepts anything with a
# .predict() method. LightGBM is a strong default if you want more
# capacity than logistic regression.

In [ ]:
# YOUR TURN: Try different models, features, targets, hyperparameters
# Ideas:
#   - Random Forest, XGBoost, neural network
#   - Add interaction features (e.g., user_aov * product_price)
#   - Train separate CTR and CVR models, combine them
#   - Use regression on revenue instead of classification
#   - Weight samples by revenue potential



## 2.5 Evaluate on the right metric

AUC measures ranking quality. But we care about revenue. Let's evaluate properly.

In [ ]:
# AUC measures ranking on click — but you're scored on revenue per
# impression. A better local check: take the top N% of your test set
# by your model's score, and compare their mean revenue to the rest.
# A useful model should put high-revenue impressions near the top.

---

# Phase 3: Optimize for Production (30 min)

Your model must score ads in real-time. The simulation has a **latency budget** (default: 50ms).
If your model is slower, some of your traffic gets a random ad choice instead — penalizing slow models.

## 3.1 Benchmark your model's latency

In [ ]:
def benchmark_latency(model, X_sample, n_trials=100):
    """Measure inference latency for a batch of 10 rows (simulates one request)."""
    batch = X_sample.head(10)  # The simulation scores 10 candidate ads per request
    latencies = []

    for _ in range(n_trials):
        start = time.perf_counter()
        if hasattr(model, 'predict_proba'):
            model.predict_proba(batch)
        else:
            model.predict(batch)
        elapsed_ms = (time.perf_counter() - start) * 1000
        latencies.append(elapsed_ms)

    latencies = np.array(latencies)
    print(f"Latency (batch of 10):")
    print(f"  Median: {np.median(latencies):.2f} ms")
    print(f"  P95:    {np.percentile(latencies, 95):.2f} ms")
    print(f"  P99:    {np.percentile(latencies, 99):.2f} ms")
    print(f"  Max:    {np.max(latencies):.2f} ms")

    budget = 50  # ms
    violations = (latencies > budget).mean()
    print(f"\n  Budget violations (>{budget}ms): {violations:.1%}")
    if violations > 0.1:
        print(f"  WARNING: >10% of requests will be penalized!")
    return latencies

print("Logistic Regression:")
benchmark_latency(lr, X_test)
print("\nLightGBM:")
benchmark_latency(lgb_model, X_test)

In [ ]:
# If your model is too slow, try:
#   - Fewer trees / smaller ensemble
#   - Fewer features (drop low-importance ones)
#   - Simpler model (logistic regression is very fast)
#   - Quantize or compress
#
# Feature importance (for feature selection):
if hasattr(lgb_model, 'feature_importances_'):
    importance = pd.Series(
        lgb_model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=True)
    importance.tail(20).plot.barh(figsize=(10, 6), color='#6c63ff')
    plt.title('Top 20 Feature Importances')
    plt.tight_layout()
    plt.show()

## 3.2 Wrap your model for deployment

The simulator calls `model.predict(features_df)` where `features_df` has the same columns as your training data.
The prediction should be a **score** — higher = better ad to show. This could be:
- P(click) from a CTR model
- P(click) * P(convert|click) from two models combined
- Predicted revenue
- Any scoring function you design

In [ ]:
class ScoringModel:
    """Wraps your trained model(s) into a scoring function for the simulator."""

    def __init__(self, model):
        # You can store multiple models here if you want a composite
        # score (e.g. self.ctr_model, self.cvr_model).
        self.model = model

    def predict(self, X):
        # Return a 1D array of scores — higher = better ad to show.
        # If `model` is a classifier, use predict_proba(X)[:, 1].
        return self.model.predict_proba(X)[:, 1]


final_model = ScoringModel(lr)  # or whatever you trained

# Verify it returns sensible scores
test_scores = final_model.predict(X_test.head(10))
print(f"Sample scores: {test_scores}")

In [ ]:
# Final latency check
print("Final model latency:")
benchmark_latency(final_model, X_test)

---

# Phase 4: Deploy & Compete! (45 min)

## 4.1 Save and upload your model

In [ ]:
from pathlib import Path

# Save model using cloudpickle (captures class definitions for the server)
model_path = f"model_{TEAM_NAME}.pkl"
with open(model_path, 'wb') as f:
    cloudpickle.dump(final_model, f)

print(f"Model saved to {model_path}")
print(f"Size: {Path(model_path).stat().st_size / 1024:.1f} KB")

In [ ]:
from pathlib import Path

# Upload to the server
with open(model_path, 'rb') as f:
    resp = requests.post(
        f"{SERVER}/api/teams/{TEAM_NAME}/model",
        files={"model": (model_path, f, "application/octet-stream")}
    )

result = resp.json()
print(f"Upload result: {result}")
if resp.ok:
    print(f"\nModel uploaded successfully!")
    print(f"  Type: {result.get('model_type')}")
    print(f"  Validation prediction: {result.get('validation_prediction'):.4f}")
    print(f"  Validation latency: {result.get('validation_latency_ms'):.2f} ms")
else:
    print(f"\nERROR: {result}")

## 4.2 Watch the live dashboard

Open the dashboard in your browser to watch the A/B test in real time:

**Dashboard URL**: `{SERVER}/dashboard`

The instructor will start the simulation once all teams have uploaded their models.

In [ ]:
# Check the leaderboard from here
resp = requests.get(f"{SERVER}/api/leaderboard")
lb = resp.json()

print(f"Simulation running: {lb['running']}")
print(f"Total requests: {lb['total_requests']:,}")
print(f"\nLeaderboard:")
for i, team in enumerate(lb['leaderboard']):
    print(f"  #{i+1} {team['team']:<20} "
          f"Rev/Impr: ${team['revenue_per_impression']:.4f}  "
          f"Revenue: ${team['revenue']:.2f}  "
          f"CTR: {team['ctr']:.2%}  "
          f"Latency: {team['avg_latency_ms']:.1f}ms")